In [1]:
from lfr_generator import generate_features_from_communities

generate_features_from_communities(true_labels=[0, 1, 2, 3, 0, 2, 1, 1, 2, 3], 
                                   mode = 'gaussian', num_features=5, n_clusters_f=5)

array([[ 0.97125674, -0.76460365,  3.03460073,  0.61124548,  0.71385807],
       [-0.02347035, -1.90776535,  1.54931914, -2.94077968, -3.37411828],
       [-2.13268081, -1.18816213, -0.1601611 , -2.95068314, -2.79929842],
       [ 2.07711242, -0.32439875, -0.05932338, -1.26684347, -0.2444996 ],
       [ 0.68870531, -1.24856614, -0.46146476,  3.41166551, -0.9437072 ],
       [-2.32058867, -2.63486623, -0.58625996, -1.43831495, -1.86405897],
       [ 0.08508547,  0.08766473,  1.17408875, -3.52106448, -3.61112272],
       [-2.31081628,  0.29297938,  0.24135883, -3.7242819 , -1.80792898],
       [-1.87211851, -0.91947954,  0.84085373, -2.61057371, -2.52198551],
       [ 3.51154167,  0.40453944,  1.61239653, -2.12955499,  2.42939772]])

## LFR graph generation

In [2]:
from lfr_generator import generate_lfr_graph

num_features = 16
n_f = 13
G, data, true_labels = generate_lfr_graph(mu=0.1, n=1000, num_features=num_features, feature_mode='gaussian', n_clusters_f=n_f)
# G_random, data_random, true_labels = generate_lfr_graph(mu=0.1, n=1000, num_features=num_features, feature_mode='random')
print("Graph generated with the following parameters:")
print(f"Number of nodes: {G.number_of_nodes()}")
print(f"Number of edges: {G.number_of_edges()}")
print(f"Number of communities: {len(set(true_labels))}")



Graph generated with the following parameters:
Number of nodes: 1000
Number of edges: 17930
Number of communities: 14


In [3]:
import torch
import dmon

def train_model(data, true_labels, num_features):
    """
    Train the DMoN model on the provided data.
    
    Parameters:
    - data: The graph data containing node features and edge indices.
    - true_labels: The ground truth labels for the nodes.
    - num_features: The number of features per node.
    """
    # Initialize the DMoN model
    import dmon
    import torch

    num_clusters = len(set(true_labels))
    model = dmon.DMoN(in_channels=num_features, num_clusters=num_clusters, gcn_skip=True)

    learning_rate = 0.005
    weight_decay = 5e-4
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    # Train the model
    for epoch in range(201):
        model.train()
        optimizer.zero_grad()
        ca, loss = model(data.x, data.edge_index)
        loss.backward()
        optimizer.step()
        # if epoch % 20 == 0:
        #     print(f"Epoch {epoch:03d} | Loss: {loss.item():.4f}")
    model.eval()
    ca, _ = model(data.x, data.edge_index)
    pred_labels = ca.argmax(dim=1).numpy()
    from sklearn.metrics import normalized_mutual_info_score
    from clusim.clustering import Clustering
    from clusim.sim import element_sim # Element-centric similarity from A.J. Gates and YY Ahn

    nmi = normalized_mutual_info_score(true_labels, pred_labels)
    print(f"NMI between DMoN and LFR ground truth: {nmi:.4f}")


    # Convert label lists to CluSim Clustering objects
    true_clustering = Clustering().from_membership_list(true_labels)
    pred_clustering = Clustering().from_membership_list(pred_labels)

    # Compute element-centric similarity
    ecs_sim = element_sim(true_clustering, pred_clustering)
    print(f"ECS between DMoN and LFR ground truth: {ecs_sim:.4f}")

In [ ]:
from lfr_generator import generate_lfr_graph

num_features = 16
n_clusters = [2, 5, 7, 14, 28, 30]
sigma_c = [0.1, 0.5, 1, 2, 5, 10]
for n in n_clusters:
    for s in sigma_c:
        G, data, true_labels = generate_lfr_graph(mu=0.1, n=1000, num_features=num_features, 
                                                  feature_mode='gaussian', n_clusters_f=n, sigma_c=s)
        print(f"Generated LFR graph with {n} clusters and sigma_c={s}.")
        train_model(data, true_labels, num_features)

Generated LFR graph with 2 clusters and sigma_c=1.
NMI between DMoN and LFR ground truth: 0.4004
ECS between DMoN and LFR ground truth: 0.2112
Generated LFR graph with 2 clusters and sigma_c=3.
NMI between DMoN and LFR ground truth: 0.2886
ECS between DMoN and LFR ground truth: 0.1403
Generated LFR graph with 2 clusters and sigma_c=5.
NMI between DMoN and LFR ground truth: 0.2914
ECS between DMoN and LFR ground truth: 0.1519
Generated LFR graph with 2 clusters and sigma_c=10.
NMI between DMoN and LFR ground truth: 0.2860
ECS between DMoN and LFR ground truth: 0.1451
Generated LFR graph with 5 clusters and sigma_c=1.
NMI between DMoN and LFR ground truth: 0.6655
ECS between DMoN and LFR ground truth: 0.3831
Generated LFR graph with 5 clusters and sigma_c=3.
NMI between DMoN and LFR ground truth: 0.6730
ECS between DMoN and LFR ground truth: 0.3677
Generated LFR graph with 5 clusters and sigma_c=5.
NMI between DMoN and LFR ground truth: 0.6392
ECS between DMoN and LFR ground truth: 0.356

In [ ]:
model.eval()
ca, _ = model(data.x, data.edge_index)
pred_labels = ca.argmax(dim=1).numpy()

# print(f"Shape of predicted labels: {pred_labels.shape}")

In [ ]:
from sklearn.metrics import normalized_mutual_info_score
from clusim.clustering import Clustering
from clusim.sim import element_sim # Element-centric similarity from A.J. Gates and YY Ahn

nmi = normalized_mutual_info_score(true_labels, pred_labels)
print(f"NMI between DMoN and LFR ground truth: {nmi:.4f}")


# Convert label lists to CluSim Clustering objects
true_clustering = Clustering().from_membership_list(true_labels)
pred_clustering = Clustering().from_membership_list(pred_labels)

# Compute element-centric similarity
ecs_sim = element_sim(true_clustering, pred_clustering)
print(f"ECS between DMoN and LFR ground truth: {ecs_sim:.4f}")


NMI between DMoN and LFR ground truth: 0.9796
ECS between DMoN and LFR ground truth: 0.9311


### Random features

In [ ]:
G_random, data_random, true_labels = generate_lfr_graph(mu=0.1, n=1000, num_features=num_features, feature_mode='random')
print("Graph generated with the following parameters:")
print(f"Number of nodes: {G_random.number_of_nodes()}")
print(f"Number of edges: {G_random.number_of_edges()}")
print(f"Number of communities: {len(set(true_labels))}")

Graph generated with the following parameters:
Number of nodes: 1000
Number of edges: 18114
Number of communities: 13


In [ ]:
import torch
import dmon

num_clusters = len(set(true_labels))
model = dmon.DMoN(in_channels=num_features, num_clusters=num_clusters, gcn_skip=True)

learning_rate = 0.005
weight_decay = 5e-4
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

# Train the model
for epoch in range(201):
    model.train()
    optimizer.zero_grad()
    ca, loss = model(data_random.x, data_random.edge_index)
    loss.backward()
    optimizer.step()
    if epoch % 20 == 0:
        print(f"Epoch {epoch:03d} | Loss: {loss.item():.4f}")

Epoch 000 | Loss: 1.2026
Epoch 020 | Loss: 0.3920
Epoch 040 | Loss: 0.0732
Epoch 060 | Loss: 0.0716
Epoch 080 | Loss: 0.0286
Epoch 100 | Loss: 0.0018
Epoch 120 | Loss: 0.0427
Epoch 140 | Loss: 0.0163
Epoch 160 | Loss: -0.0056
Epoch 180 | Loss: -0.0168
Epoch 200 | Loss: -0.0279


In [ ]:
model.eval()
ca, _ = model(data_random.x, data_random.edge_index)
pred_labels = ca.argmax(dim=1).numpy()

# print(f"Shape of predicted labels: {pred_labels.shape}")

In [ ]:
from sklearn.metrics import normalized_mutual_info_score
from clusim.clustering import Clustering
from clusim.sim import element_sim # Element-centric similarity from A.J. Gates and YY Ahn

nmi = normalized_mutual_info_score(true_labels, pred_labels)
print(f"NMI between DMoN and LFR ground truth: {nmi:.4f}")


# Convert label lists to CluSim Clustering objects
true_clustering = Clustering().from_membership_list(true_labels)
pred_clustering = Clustering().from_membership_list(pred_labels)

# Compute element-centric similarity
ecs_sim = element_sim(true_clustering, pred_clustering)
print(f"ECS between DMoN and LFR ground truth: {ecs_sim:.4f}")


NMI between DMoN and LFR ground truth: 0.4929
ECS between DMoN and LFR ground truth: 0.3502
